# Evaluate LSTM Test Submission On Secret Set

This notebook evaluates the local `model_info/test` LSTM bundle on the TOML-defined `secret_set_1` dataset and logs the full backtest to MLflow using the shared toolkit helpers.

## Setup

Change `submission_dir`, `dataset_name`, or `run_name` to reuse this pattern for another submitted bundle later.

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import math
import os

mpl_cache_dir = Path('/private/tmp/portfolio_optimizer_mpl_cache')
mpl_cache_dir.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(mpl_cache_dir))

import mlflow
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

from portfolio_toolkit import (
    backtest_weights,
    build_features,
    get_dataset_spec,
    init_mlflow,
    load_prices,
    log_backtest,
    log_model_submission,
    log_portfolio,
    log_predictions,
    split_dates,
    start_run,
    validate_prediction_frame,
    validate_weights_frame,
    weights_from_predictions_rank_long_only,
    write_backtest_artifacts,
)

repo_root = Path.cwd().resolve()
if not (repo_root / 'configs' / 'datasets.toml').exists():
    repo_root = Path('../../').resolve()

dataset_name = 'secret_set_1'
submission_dir = repo_root / 'model_info' / 'test'
manifest_path = submission_dir / 'manifest.json'
source_notebook = repo_root / 'notebooks' / 'templates' / 'lstm_robust_submission_workflow.ipynb'

manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
checkpoint_relative = manifest.get('artifact_map', {}).get('torch_checkpoint', 'lstm_model.pt')
checkpoint_path = submission_dir / checkpoint_relative
if not checkpoint_path.exists():
    checkpoint_path = submission_dir / Path(checkpoint_relative).name

model_name = manifest.get('model_name', 'lstm_submission')
run_name = f'{model_name}_{dataset_name}_full_backtest'
output_dir = repo_root / 'runs' / run_name
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Repo root:', repo_root)
print('Dataset:', dataset_name)
print('Submission dir:', submission_dir)
print('Checkpoint:', checkpoint_path)
print('Run name:', run_name)
print('Torch device:', device)


/Users/adamthorne/.pyenv/versions/3.12.7/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Repo root: /Users/adamthorne/Desktop/Portfolio/Portfolio-Optimizer
Dataset: secret_set_1
Submission dir: /Users/adamthorne/Desktop/Portfolio/Portfolio-Optimizer/model_info/test
Checkpoint: /Users/adamthorne/Desktop/Portfolio/Portfolio-Optimizer/model_info/test/lstm_model.pt
Run name: robust_lstm_submission_secret_set_1_full_backtest
Torch device: cpu


## LSTM Submission Implementation

These functions mirror the implementation notebook's required inference surface: `build_model_features`, `load_submission_model`, and `predict_from_prices`.

In [2]:
checkpoint_preview = torch.load(checkpoint_path, map_location='cpu')
metadata = checkpoint_preview['metadata']
feature_names = list(metadata['feature_names'])
base_feature_names = list(metadata.get('base_feature_names', feature_names))
custom_feature_names = list(metadata.get('custom_feature_names', []))
model_config = dict(metadata['config'])
preprocessing = dict(metadata['preprocessing'])
horizon = int(model_config.get('horizon', manifest.get('horizon', 5)))
target_col = str(model_config.get('target', manifest.get('target', f'forward_return_{horizon}d')))

def build_model_features(prices: pd.DataFrame) -> pd.DataFrame:
    features = build_features(prices, feature_names=base_feature_names)
    if 'vol_ratio_5_20' in custom_feature_names:
        features['vol_ratio_5_20'] = features['vol_5d'] / features['vol_20d'].replace(0.0, np.nan)
    if 'momentum_5_20_spread' in custom_feature_names:
        features['momentum_5_20_spread'] = features['momentum_5d'] - features['momentum_20d']
    if 'signed_volume_pressure' in custom_feature_names:
        features['signed_volume_pressure'] = features['return_1d'] * features['volume_zscore_20d']
    missing = sorted(set(feature_names) - set(features.columns))
    if missing:
        raise ValueError(f'Missing model features: {missing}')
    return features.replace([np.inf, -np.inf], np.nan)

def apply_feature_preprocessing(
    frame: pd.DataFrame,
    preprocessing: dict,
    feature_order: list[str],
) -> pd.DataFrame:
    prepared = frame.copy()
    low = pd.Series(preprocessing['feature_clip_low'])
    high = pd.Series(preprocessing['feature_clip_high'])
    center = pd.Series(preprocessing['feature_center'])
    scale = pd.Series(preprocessing['feature_scale']).replace(0.0, 1.0)
    prepared[feature_order] = prepared[feature_order].clip(lower=low, upper=high, axis=1)
    prepared[feature_order] = (prepared[feature_order] - center[feature_order]) / scale[feature_order]
    prepared[feature_order] = prepared[feature_order].replace([np.inf, -np.inf], np.nan)
    return prepared

class WindowedInferenceDataset(Dataset):
    def __init__(self, arrays: dict[str, np.ndarray], windows: list[tuple[str, int, int, int]]):
        self.arrays = arrays
        self.windows = windows

    def __len__(self) -> int:
        return len(self.windows)

    def __getitem__(self, index: int):
        ticker, start_pos, end_pos, candidate_index = self.windows[index]
        x = self.arrays[ticker][start_pos : end_pos + 1]
        return torch.from_numpy(x), torch.tensor(candidate_index, dtype=torch.long)

def build_inference_windows(
    frame: pd.DataFrame,
    feature_order: list[str],
    lookback: int,
    candidate_key_to_index: dict[tuple[pd.Timestamp, str], int],
) -> WindowedInferenceDataset:
    arrays = {}
    windows = []
    ordered = frame.sort_values(['ticker', 'date']).reset_index(drop=True)
    for ticker, group in ordered.groupby('ticker', sort=False):
        values = group[feature_order].to_numpy(dtype=np.float32)
        dates = pd.to_datetime(group['date'], utc=True).dt.tz_localize(None).to_numpy()
        valid_features = np.isfinite(values).all(axis=1)
        arrays[ticker] = values
        for end_pos in range(lookback - 1, len(group)):
            key = (pd.Timestamp(dates[end_pos]), str(ticker).upper())
            candidate_index = candidate_key_to_index.get(key)
            if candidate_index is None:
                continue
            start_pos = end_pos - lookback + 1
            if valid_features[start_pos : end_pos + 1].all():
                windows.append((ticker, start_pos, end_pos, candidate_index))
    return WindowedInferenceDataset(arrays, windows)

class LSTMReturnModel(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, num_layers: int, dropout: float):
        super().__init__()
        lstm_dropout = dropout if num_layers > 1 else 0.0
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=lstm_dropout,
        )
        self.norm = nn.LayerNorm(hidden_dim)
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, max(8, hidden_dim // 2)),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(max(8, hidden_dim // 2), 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        _, (hidden, _) = self.lstm(x)
        last_hidden = hidden[-1]
        return self.head(self.norm(last_hidden)).squeeze(-1)

def load_submission_model(checkpoint_path: str | Path, map_location=None) -> dict:
    checkpoint = torch.load(checkpoint_path, map_location=map_location or 'cpu')
    metadata = checkpoint['metadata']
    config = metadata['config']
    loaded_model = LSTMReturnModel(
        input_dim=len(metadata['feature_names']),
        hidden_dim=int(config['hidden_dim']),
        num_layers=int(config['num_layers']),
        dropout=float(config['dropout']),
    )
    loaded_model.load_state_dict(checkpoint['model_state_dict'])
    loaded_model.eval()
    return {'model': loaded_model, 'metadata': metadata}

def normalize_optional_dates(dates) -> set[pd.Timestamp] | None:
    if dates is None:
        return None
    if isinstance(dates, (str, pd.Timestamp, np.datetime64)):
        dates = [dates]
    normalized = pd.to_datetime(pd.Series(list(dates)), utc=True).dt.tz_localize(None)
    return {pd.Timestamp(value) for value in normalized}

def normalize_optional_tickers(tickers) -> list[str] | None:
    if tickers is None:
        return None
    if isinstance(tickers, str):
        tickers = [tickers]
    return [str(ticker).upper() for ticker in tickers]

def predict_from_prices(model_bundle: dict, prices: pd.DataFrame, dates=None, tickers=None) -> pd.DataFrame:
    model = model_bundle['model']
    metadata = model_bundle['metadata']
    feature_order = metadata['feature_names']
    config = metadata['config']
    preprocessing = metadata['preprocessing']
    benchmark_ticker = config.get('benchmark_ticker', 'SPY')
    vol_col = config.get('vol_feature_col', 'vol_20d')

    features = build_model_features(prices)
    features['date'] = pd.to_datetime(features['date'], utc=True).dt.tz_localize(None)
    features['ticker'] = features['ticker'].astype(str).str.upper()

    requested_dates = normalize_optional_dates(dates)
    requested_tickers = normalize_optional_tickers(tickers)
    if requested_tickers is None:
        requested_tickers = sorted(set(features['ticker']) - {benchmark_ticker})

    candidate_cols = ['date', 'ticker', vol_col, 'momentum_20d']
    candidate = features.loc[features['ticker'].isin(requested_tickers), candidate_cols].copy()
    if requested_dates is not None:
        candidate = candidate.loc[candidate['date'].isin(requested_dates)].copy()
    candidate = candidate.drop_duplicates(['date', 'ticker']).sort_values(['date', 'ticker']).reset_index(drop=True)
    if candidate.empty:
        return pd.DataFrame(columns=['date', 'ticker', 'horizon', 'expected_return', 'expected_volatility', 'uncertainty'])

    fallback = pd.to_numeric(candidate['momentum_20d'], errors='coerce').fillna(0.0)
    fallback = fallback.clip(preprocessing['target_clip_low'], preprocessing['target_clip_high'])
    predictions = candidate[['date', 'ticker']].copy()
    predictions['horizon'] = int(config['horizon'])
    predictions['expected_return'] = fallback.to_numpy(dtype=float) * float(preprocessing['fallback_momentum_weight'])
    predictions['uncertainty'] = 0.95

    vol = pd.to_numeric(candidate[vol_col], errors='coerce') * math.sqrt(252.0)
    vol = vol.replace([np.inf, -np.inf], np.nan).clip(lower=preprocessing['volatility_floor'])
    predictions['expected_volatility'] = vol.fillna(preprocessing['volatility_fallback']).to_numpy(dtype=float)

    scaled_features = apply_feature_preprocessing(features, preprocessing, feature_order)
    candidate_key_to_index = {
        (pd.Timestamp(row['date']), str(row['ticker']).upper()): int(idx)
        for idx, row in candidate.iterrows()
    }
    inference_dataset = build_inference_windows(
        scaled_features.loc[scaled_features['ticker'].isin(requested_tickers)].copy(),
        feature_order,
        int(config['lookback']),
        candidate_key_to_index,
    )

    if len(inference_dataset) > 0:
        model_device = next(model.parameters()).device
        loader = DataLoader(inference_dataset, batch_size=2048, shuffle=False, num_workers=0)
        model.eval()
        filled_indexes = []
        filled_values = []
        with torch.no_grad():
            for x_batch, index_batch in loader:
                x_batch = x_batch.to(model_device)
                scaled_pred = model(x_batch).detach().cpu().numpy()
                raw_pred = scaled_pred * preprocessing['target_scale'] + preprocessing['target_center']
                raw_pred = raw_pred * preprocessing['prediction_shrinkage']
                raw_pred = np.clip(raw_pred, preprocessing['target_clip_low'], preprocessing['target_clip_high'])
                filled_indexes.extend(index_batch.numpy().tolist())
                filled_values.extend(raw_pred.tolist())
        predictions.loc[filled_indexes, 'expected_return'] = np.asarray(filled_values, dtype=float)
        predictions.loc[filled_indexes, 'uncertainty'] = 0.50

    predictions = predictions.replace([np.inf, -np.inf], np.nan)
    predictions['expected_return'] = predictions['expected_return'].fillna(0.0)
    predictions['expected_volatility'] = predictions['expected_volatility'].fillna(preprocessing['volatility_fallback'])
    predictions['uncertainty'] = predictions['uncertainty'].fillna(1.0)
    return predictions.sort_values(['date', 'ticker', 'horizon']).reset_index(drop=True)

print('Feature count:', len(feature_names))
print('Horizon:', horizon)
print('Target:', target_col)


Feature count: 24
Horizon: 5
Target: forward_return_5d


## Predict And Backtest

In [9]:
spec = get_dataset_spec(dataset_name, repo_root=repo_root)
splits = split_dates(dataset_name, repo_root=repo_root)
prices = load_prices(dataset_name, refresh=True, repo_root=repo_root)
prices['date'] = pd.to_datetime(prices['date'], utc=True).dt.tz_localize(None)
test_start, test_end = splits["test"]

trading_dates = pd.DatetimeIndex(
    sorted(prices.loc[
        (prices["date"] >= test_start) & (prices["date"] <= test_end),
        "date"
    ].unique())
)

# Start date, then every 5 trading sessions
rebalance_dates = trading_dates[::5]


submission_bundle = load_submission_model(checkpoint_path, map_location=device)
submission_bundle['model'].to(device)

raw_predictions = predict_from_prices(
    submission_bundle,
    prices,
    dates=rebalance_dates,
    tickers=spec.tickers,
)

predictions = validate_prediction_frame(raw_predictions, dataset_name=dataset_name, horizon=horizon, repo_root=repo_root)

portfolio = weights_from_predictions_rank_long_only(
    predictions,
    score_column='expected_return',
    dataset_name=dataset_name,
    strategy_name=model_name,
)
portfolio.weights = validate_weights_frame(portfolio.weights, dataset_name=dataset_name, repo_root=repo_root)

result = backtest_weights(
    dataset_name,
    portfolio,
    benchmark=spec.default_benchmark,
    repo_root=repo_root,
)

artifact_paths = write_backtest_artifacts(result, output_dir / 'backtest')

assert not predictions.empty
assert result.metrics['evaluation_trading_days'] > 0
assert Path(artifact_paths['quantstats_report']).exists()

print('Downloaded/loaded price rows:', len(prices))
print('Predictions:', predictions.shape)
print('Weights:', portfolio.weights.shape)
print(json.dumps(result.metrics, indent=2, sort_keys=True))
print('Artifacts:', json.dumps(artifact_paths, indent=2, sort_keys=True))


Downloaded/loaded price rows: 67625
Predictions: (4422, 6)
Weights: (201, 22)
{
  "annual_excess_return_vs_benchmark": 0.024184427085688354,
  "annual_return": 0.18310642208954486,
  "annual_volatility": 0.27467366095443424,
  "average_turnover": 0.17155330069022479,
  "benchmark_annual_return": 0.1589219950038565,
  "benchmark_annual_volatility": 0.2584388831901583,
  "benchmark_max_drawdown": -0.2604372562192223,
  "benchmark_sharpe": 0.6149306677158265,
  "benchmark_total_return": 0.8017341442390462,
  "calmar": 0.663280494711831,
  "evaluation_trading_days": 1003.0,
  "evaluation_years": 3.991786447638604,
  "excess_return_vs_benchmark": 0.1548361802125946,
  "max_drawdown": -0.2760618223352057,
  "sharpe": 0.6666326194265874,
  "sharpe_vs_benchmark": 0.05170195171076086,
  "sortino": 0.8943817920427258,
  "total_return": 0.9565703244516408
}
Artifacts: {
  "benchmarks": "/Users/adamthorne/Desktop/Portfolio/Portfolio-Optimizer/runs/robust_lstm_submission_secret_set_1_full_backtest/

## Log To MLflow

In [10]:
mlflow_layout = init_mlflow(repo_root)
print('MLflow tracking URI:', mlflow_layout['tracking_uri'])

with start_run(
    run_name=run_name,
    dataset_name=dataset_name,
    tags={
        'workflow': 'evaluate_lstm_test_submission_on_secret_set',
        'model_family': manifest.get('model_family', 'torch_lstm'),
        'prediction_horizon': str(horizon),
        'source_submission_dir': str(submission_dir.relative_to(repo_root)),
    },
    repo_root=repo_root,
) as run:
    mlflow.log_params({
        'model_name': model_name,
        'dataset_name': dataset_name,
        'dataset_label': spec.name,
        'ticker_count': len(spec.tickers),
        'horizon': horizon,
        'feature_count': len(feature_names),
        'lookback': int(model_config['lookback']),
        'hidden_dim': int(model_config['hidden_dim']),
        'dropout': float(model_config['dropout']),
        'portfolio_builder': 'weights_from_predictions_rank_long_only',
    })
    mlflow.log_dict(manifest, artifact_file='input_submission/manifest.json')
    log_predictions(predictions)
    log_portfolio(portfolio)
    log_backtest(result)
    logged_manifest = log_model_submission(
        {'torch_checkpoint': checkpoint_path},
        model_name=model_name,
        model_family=manifest.get('model_family', 'torch_lstm'),
        feature_names=feature_names,
        target=target_col,
        horizon=horizon,
        preprocessing=preprocessing,
        model_config={
            **model_config,
            'architecture': 'LSTMReturnModel',
            'input_dim': len(feature_names),
            'required_functions': ['build_model_features', 'predict_from_prices'],
            'optional_functions': ['load_submission_model'],
            'portfolio_builder': 'weights_from_predictions_rank_long_only',
            'no_ticker_embeddings': True,
        },
        source_files=[source_notebook],
        notes=manifest.get('notes'),
    )
    run_id = run.info.run_id

print('MLflow run_id:', run_id)
print(json.dumps(logged_manifest, indent=2, sort_keys=True)[:2000])


MLflow tracking URI: https://adams-macbook-pro.tail5ddc35.ts.net
🏃 View run robust_lstm_submission_secret_set_1_full_backtest at: https://adams-macbook-pro.tail5ddc35.ts.net/#/experiments/3/runs/a365998ab62f43edb4b7673708f10faa
🧪 View experiment at: https://adams-macbook-pro.tail5ddc35.ts.net/#/experiments/3
MLflow run_id: a365998ab62f43edb4b7673708f10faa
{
  "artifact_files": [
    "artifacts/lstm_model.pt"
  ],
  "artifact_map": {
    "torch_checkpoint": "artifacts/lstm_model.pt"
  },
  "feature_names": [
    "return_1d",
    "return_5d",
    "log_return_1d",
    "vol_5d",
    "vol_20d",
    "downside_vol_20d",
    "momentum_5d",
    "momentum_20d",
    "momentum_60d",
    "price_to_sma_20d",
    "price_to_sma_50d",
    "price_to_ema_12d",
    "price_to_ema_26d",
    "rsi_14",
    "bollinger_z_20d",
    "volume_zscore_20d",
    "dollar_volume_ratio_20d",
    "intraday_range",
    "close_open_gap",
    "excess_return_20d_vs_spy",
    "relative_momentum_20d_vs_spy",
    "vol_ratio_5_20